[Reference](https://medium.com/@sauravsinghsisodiya/delta-lake-on-azure-production-lessons-on-optimize-vacuum-schema-evolution-and-synapse-vs-databricks-5ec6c7d21dc7$0)

# OPTIMIZE and VACUUM are not free, and timing matters more than you think

In [1]:
# this will run forever on a large table
spark.sql("OPTIMIZE my_table")

# this is what you actually want
spark.sql("OPTIMIZE my_table WHERE event_date >= '2025-01-01'")

In [2]:
# 7 days is the default — don't go lower without checking active streams first
spark.sql("VACUUM my_table RETAIN 168 HOURS")

# never run VACUUM while streaming jobs are actively writing to the same table

# Schema evolution works until it doesn’t


In [3]:
# this does NOT rename the column in historical records
df = df.withColumnRenamed("old_name", "new_name")
df.write.option("mergeSchema", "true").format("delta").mode("append").save(path)

# historical records: "old_name"
# new records: "new_name"
# queries expecting one or the other will break in non-obvious ways

In [4]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)

# Small files keep coming back

In [5]:
# enable these for streaming tables — they help
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

# ZORDER on columns you actually filter on - not columns that seem important
spark.sql("OPTIMIZE my_table ZORDER BY (customer_id, event_date)")

# Delta on Synapse and Delta on Databricks are not the same thing


```sql
-- reading Delta in Synapse serverless: works
SELECT * FROM
OPENROWSET(
    BULK 'https://yourstorage.dfs.core.windows.net/container/delta_table/',
    FORMAT = 'DELTA'
) AS result

-- UPDATE, DELETE, MERGE against Delta in Synapse serverless: none of these work
```

# Time travel works until VACUUM runs


In [6]:
# by version
spark.read.format("delta").option("versionAsOf", 10).load(path)

# by timestamp
spark.read.format("delta").option("timestampAsOf", "2025-12-01").load(path)

In [7]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql("VACUUM my_table RETAIN 720 HOURS")  # 30 days

# or set it at table level so it persists
ALTER TABLE my_table SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 30 days')

# What a production Delta setup actually looks like

In [8]:
# auto-optimize — sensible default for most tables
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

# set retention before VACUUM ever runs, not after
ALTER TABLE my_table SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 30 days')

# enable CDF if anything downstream needs incremental reads
ALTER TABLE my_table SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')